# Project Task: Compositional Image Retrieval
This Notebook explores the task of compositional image retrieval, where the goal is to retrieve images based on a combination of visual and textual inputs. 

We present and evaluate different approaches to this task, comparing their performance with a baseline method.

## Environment Setup

This notebook is designed for Google Colab.

Before running, ensure:
- Google Drive is mounted.
- The dataset zip exists at `/content/drive/MyDrive/datasets/celeba.zip`.
- You run cells top-to-bottom at least once to initialize all variables.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Create the directory if it doesn't exist
!mkdir -p /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

### Imports and Utilities

In [ ]:
import torch
from pathlib import Path
import os
import json
from typing import Callable
from functools import partial


import matplotlib.pyplot as plt
import numpy as np

from transformers import CLIPModel, CLIPProcessor
from torchvision.datasets import CelebA
import torchvision

In [ ]:
def plot_images(celeba_dataset: object, indices: list[int], n_cols: int, n_rows: int, figsize: tuple[int, int]=(20, 10)):
    """Utility function to plot a grid of images given their indices in the CelebA dataset.
    Args:
        celeba_dataset: The CelebA dataset object.
        indices: A list of indices corresponding to the images to be plotted.
        n_cols: Number of columns in the grid.
        n_rows: Number of rows in the grid.
        figsize: Size of the figure (width, height).
    """
    if len(indices) > n_cols * n_rows:
        raise ValueError("Number of indices exceeds the grid capacity")
    
    _, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    for counter, img_idx in enumerate(indices):
        img, _ = celeba_dataset[img_idx]
        if n_rows == 1:
            ax = axes[counter % n_cols]
        else:
            ax = axes[counter // n_cols, counter % n_cols]
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_metrics_across_k(average_results_per_query: list[dict], title: str = "Retrieval Metrics across K"):
    """
    Plot Recall@K and Precision@K as a grouped bar chart: one bar per query, grouped by K, with 95% confidence intervals.
    Args:
        average_results_per_query: Output from compute_query_average_results() for each query — list of per-query average dicts.
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    n_queries = len(average_results_per_query)
    x = np.arange(n_queries)
    width = 0.25
    offsets = [-width, 0.0, width]
    colors = [plt.cm.tab10(i) for i in range(len(k_values))]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title)

    for k, offset, color in zip(k_values, offsets, colors):
        recall_means    = [q[f"Recall@{k}"]       for q in average_results_per_query]
        recall_cis      = [q[f"Recall@{k}_CI"]    for q in average_results_per_query]
        precision_means = [q[f"Precision@{k}"]    for q in average_results_per_query]
        precision_cis   = [q[f"Precision@{k}_CI"] for q in average_results_per_query]

        ax1.bar(x + offset, recall_means,    width, yerr=recall_cis,    capsize=4, ecolor="black", color=color, label=f"K={k}")
        ax2.bar(x + offset, precision_means, width, yerr=precision_cis, capsize=4, ecolor="black", color=color, label=f"K={k}")

    for ax, metric in [(ax1, "Recall"), (ax2, "Precision")]:
        ax.set_xlabel("Query")
        ax.set_ylabel(f"{metric}@K")
        ax.set_title(f"{metric}@K per query")
        ax.set_xticks(x)
        ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        ax.legend(title="K")

    plt.tight_layout()
    plt.show()


def plot_methods_comparison(method_results: dict[str, list[dict]], title: str = "Method Comparison across queries"):
    """
    Per-query line comparison of N retrieval methods. Layout is a 2 x 3 grid:
    rows are Recall (top) and Precision (bottom); columns are K in {1, 5, 10}.
    In each subplot, one segmented line per method connects the metric values
    over the query axis, so per-query differences between methods are visible.
    Args:
        method_results: Dict mapping method name -> average_results_per_query (same shape consumed by plot_metrics_across_k).
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    method_names = list(method_results.keys())
    n_methods = len(method_names)
    if n_methods == 0:
        raise ValueError("method_results must contain at least one method.")

    n_queries = len(next(iter(method_results.values())))
    x = np.arange(n_queries)
    colors = [plt.cm.tab10(i % 10) for i in range(n_methods)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
    fig.suptitle(title)

    for row_idx, metric in enumerate(["Recall", "Precision"]):
        for col_idx, k in enumerate(k_values):
            ax = axes[row_idx, col_idx]
            for method, color in zip(method_names, colors):
                ys = [q[f"{metric}@{k}"] for q in method_results[method]]
                ax.plot(x, ys, marker="o", color=color, label=method)
            ax.set_title(f"{metric}@{k}")
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)], rotation=45, ha="right")

    for ax in axes[:, 0]:
        ax.set_ylabel("Score")
    for ax in axes[-1, :]:
        ax.set_xlabel("Query")

    axes[0, 0].legend(title="Method", loc="best")

    plt.tight_layout()
    plt.show()

## Data loading and Exploration
In this section, we load the dataset and perform an initial exploration to understand its structure and main characteristics.

The dataset used is `CelebA`, which contains 19,962 samples. Each sample consists of:
- an image of size `178 × 218`, representing a face of a celebrity;
- a set of 40 attributes that describe visual features of the person in the image.

We will briefly inspect the dataset by visualizing some samples and examining the associated attributes, in order to get a better understanding of the data before proceeding with further analysis.

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")
celeba = CelebA(root=data_root, split="test", download=False)

# This should be 19.962
print("Number of samples:", len(celeba))

# show element size
sample_img, sample_attrs = celeba[0]
print(f"Sample image size: {sample_img.size}")
print(f"Number of attributes: {len(sample_attrs)}")

### Sample visualization
First, we can visualize a random selection of images from the dataset to get a sense of the variety and quality of the images. We will display 50 random images in a grid format.

In [ ]:
# Get 50 random images and visualize them
indices = np.random.choice(len(celeba), size=50, replace=False)
plot_images(celeba, indices=indices, n_cols=10, n_rows=5)

### Attribute annotation
Now that we know how to load our dataset and we have visualized some samples, let's move to understanding how attributes are annotated in the dataset. Each image in the dataset is annotated with a set of 40 binary attributes, from the following list. 

Here, we also report how frequently each attribute appears in the dataset, which is important to understand the distribution of attributes and to design a retrieval system that can handle rare attributes effectively.


In [ ]:
all_labels = np.array([labels for _, labels in celeba])

attr_counts = all_labels.sum(axis=0)
attr_freq = all_labels.mean(axis=0)

print(f"{'Attribute':<20} {'Count':>10} {'Frequency':>10}")
print("-" * 45)

for attr, count, freq in zip(celeba.attr_names, attr_counts, attr_freq):
    print(f"{attr:<20} {count:>10} {freq:>10.3f}")

Let's define few other utilities functions that will facilitate the handling of attributes later on.
We will create a:
- `inx2attribute`: mapping from indices to attributes (e.g., index 0 corresponds to "5_o_Clock_Shadow", index 1 corresponds to "Arched_Eyebrows", etc.)

- `attribute2index`: mapping from attributes to indices (e.g., "5_o_Clock_Shadow" corresponds to index 0, "Arched_Eyebrows" corresponds to index 1, etc.)

- `retrieve_by_attributes(parameters)`: function that retrieves images based on specified attributes. This function will be crucial for our image retrieval system, allowing us to query the dataset for images that match certain attribute criteria.

- `plot_image_with_attributes`: function that displays an image along with its active attributes.

Query format for `retrieve_by_attributes`:
- Use `"+"` when the attribute must be present.
- Use `"-"` when the attribute must be absent.

Example: `{"Bald": "+", "Eyeglasses": "-"}`.

In [ ]:
# Assign a unique index to each attribute, and get the inverse mapping
idx2attribute = {idx: name for idx, name in enumerate(celeba.attr_names)}
attribute2idx = {name: idx for idx, name in enumerate(celeba.attr_names)}

def retrieve_by_attributes(parameters:dict):
    """
    Helper function that retrieve all the images that satisfy the conditions specified 
    in `parameters`.
    Params:
        - parameters: a dictionary where keys are attribute names and values are either "+" (must have the attribute) or "-" (must not have the attribute).  
    Returns:
        - A list of indices of images that satisfy the specified conditions.
    """
    # Start with all indices
    valid_indices = set(range(len(celeba)))
    
    # For each attribute condition, filter the indices
    for attr_name, value in parameters.items():
        attr_idx = attribute2idx[attr_name]
        if value == "+":
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 0:
                    valid_indices.remove(idx)
        elif value == "-":
            # Must not have the attribute
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 1:
                    valid_indices.remove(idx)
        else:
            raise ValueError(f"Invalid value for attribute condition: {value}. Use '+' or '-'.")
    
    return list(valid_indices)

def plot_image_with_attributes(idx: int, figsize: tuple[int, int]=(10, 5)):
    """Helper function to plot a single image along with its active attributes as text on the right side of the image."""
    img, labels = celeba[idx]
    active_attrs = [idx2attribute[idx] for idx, value in enumerate(labels) if value == 1]

    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=figsize)
    # Left: image
    ax_img.imshow(img)
    ax_img.axis('off')

    # Right: centered text
    ax_text.axis('off')
    text = "\n".join(active_attrs)

    ax_text.text(
        0.5, 0.5, text,
        fontsize=10,
        ha='center',   # horizontal alignment
        va='center'    # vertical alignment
    )

    plt.tight_layout()
    plt.show()

Now that we have the mapping, we can easily get the attributes of any image in the dataset. For example, let's get the attributes of a given image index.

In [ ]:
IMAGE_INDEX = 99
plot_image_with_attributes(99)

Now that we have everything in place, let's try to analyze some possible queries.


In [ ]:
query_1 = {"Bald": "+",
           "Smiling": "+",
           "Eyeglasses": "-",
           }
retrieved_images = retrieve_by_attributes(query_1)
print(f"Number of retrieved images: {len(retrieved_images)}")

# Plot up to 10 random retrieved images (without replacement).
n_samples = min(10, len(retrieved_images))
if n_samples == 0:
    print("No images match this query.")
else:
    sampled_indices = np.random.choice(retrieved_images, size=n_samples, replace=False)
    n_cols = 5
    n_rows = int(np.ceil(n_samples / n_cols))
    plot_images(celeba, indices=sampled_indices, n_cols=n_cols, n_rows=n_rows)

# Offline Feature Extraction
In this section we extract features from our dataset using a Vision Language Model (VLM).

This representation is computed once and then kept frozen offline. The goal is not to improve the VLM encoder itself, but to study and improve retrieval behavior on top of fixed CLIP embeddings.

In [ ]:
cache_dir = "/content/drive/MyDrive/datasets/clip_cache"
save_path = os.path.join(cache_dir, "embeddings.pt")

### Extracting features using a VLM
In this section we will be using the CLIP model to extract features from our dataset. We will be using the ViT-B/32 model, which is a smaller version of the original CLIP model. 

Since these embeddings are static, we can compute them offline and keep them frozen. This means that we don't have to worry about the computational cost of computing the embeddings during training, and we can also use a larger batch size for training our retriever model.

The result of this process will be a list of embeddings, where each embedding is of size 512 and corresponds to an image in our dataset. 

In [ ]:
print("Loading CLIP model...")

# Check if CUDA is available and set the device accordingly
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the CLIP model and processor, and move the model to the appropriate device
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("Model loaded.")

# Set the model to evaluation mode 
model.eval()

# Check if cached embeddings exist
if os.path.exists(save_path):
    print(f"Found cached embeddings at {save_path}")
    print("Loading cached embeddings...")
    data = torch.load(save_path)

    embeddings = data["embeddings"].to(device)
    print(f"Loaded embeddings with shape: {embeddings.shape}")
    labels = data["labels"]
    print(f"Loaded labels with shape: {labels.shape}")
# Otherwise, compute the embeddings and save them to disk
else:
    print("Cache not found. Encoding dataset...")
    all_embeddings = []
    all_labels = []

    # Disable gradient computation for efficiency, since we are only doing inference
    print("Encoding images...")
    with torch.no_grad():
        for image, label in celeba:
            # Preprocess the image using the CLIP processor
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Get the image embeddings from the model
            feats = model.get_image_features(**inputs)
            if isinstance(feats, torch.Tensor):
                image_features = feats
            elif hasattr(feats, "pooler_output") and feats.pooler_output is not None:
                image_features = feats.pooler_output
            elif isinstance(feats, tuple) and len(feats) > 0:
                image_features = feats[0]
            else:
                raise TypeError(f"Unexpected image feature output type: {type(feats)}")

            # Normalize to unit sphere before storing so dot product == cosine similarity
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            # Move to CPU and store in list
            all_embeddings.append(image_features.cpu())
            all_labels.append(label)
    print("Encoding completed.")

    # Concatenate all embeddings and labels into tensors
    embeddings = torch.cat(all_embeddings, dim=0)
    labels = torch.stack(all_labels, dim=0)

    # Save the embeddings and labels to disk for future use
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save({
        "embeddings": embeddings,
        "labels": labels
    }, save_path)

    print("Saved embeddings to disk.")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Labels shape: {labels.shape}")

### Sanity check
Now that embeddings are available, let's run a quick sanity check to verify retrieval quality.

We will pick a source image, compare its CLIP embedding against all dataset embeddings, and inspect the nearest matches.

In [ ]:
source_idx = 10006
img, _ = celeba[source_idx]

plt.figure(figsize=(4, 4))
plt.imshow(img)

Now that we have our source image and its CLIP encoding, let's find the nearest embeddings in the dataset.

We normalize embeddings to unit vectors at extraction time, so similarity between any two embeddings reduces to a plain dot product.
For unit vectors, `dot(a, b) == cosine_similarity(a, b)`, so the metric is unchanged — but retrieval becomes a single matrix multiply instead of a per-pair loop.
We exclude the source image itself from the top results.

In [ ]:
if "embeddings" not in globals():
    raise RuntimeError(
        "Embeddings not found. Run the offline feature extraction cell above first."
    )

source_embedding = embeddings[source_idx]

# Dot product == cosine similarity for unit-norm embeddings (single matrix-vector multiply)
similarities = (embeddings @ source_embedding).tolist()

# Get the 5 highest-similarity matches, excluding the source image itself
nearest_indices = torch.argsort(torch.tensor(similarities), descending=True)[1:6].tolist()
nearest_similarities = [similarities[i] for i in nearest_indices]

print("Nearest indices:", nearest_indices)
print("Nearest cosine similarities:", nearest_similarities)

We have found the nearest embeddings! Let's visualize the nearest images to our source image and see if they are indeed similar.

In [ ]:
fig, axes = plt.subplots(ncols=5, figsize=(25, 5))

for i, image_idx in enumerate(nearest_indices):
    img, labels = celeba[image_idx]
    ax = axes[i]
    
    ax.set_title(f"Cosine sim: {nearest_similarities[i]:.4f}")
    
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Metrics
In this section we will define the metrics that we will use to evaluate our retrieval system.

In particular, we will be using the following metrics:
- **Recall@K**: This metric measures if **at least one** valid truth image is present in the top K retrieved images. It is defined as:
$$Recall@K = \begin{cases} 1 & \text{if } |\mathcal{R}_K \cap \mathcal{G}| > 0 \\ 0 & \text{otherwise} \end{cases}$$
- **Precision@K**: This metric measures the proportion of relevant images in the top K retrieved images. It is defined as:
$$Precision@K = \frac{| \mathcal{R}_K \cap \mathcal{G} |}{K}$$

Where:
- $\mathcal{R}_K$ is the set of top K retrieved images for the given query.
- $\mathcal{G}$ is the set of ground truth relevant images for the given query.



In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
) -> dict:
    """
    Evaluate the retrieval performance for a single source image.
    Args:
        - retrieved_indices: list of image IDs predicted by the model, ordered by similarity (descending).
        - ground_truth_indices: list of valid target IDs from the benchmark JSON.
        - k: the cutoff for top-K evaluation (e.g., 1, 5, 10).
    Return:
        - A dictionary containing Recall@K and Precision@K metrics.

    """
    # Get the top K retrieved indices
    #NOTE: the retrieved_indices must be ordered by similarity in descending order
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

In [ ]:
# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

## Evaluation Protocol
To assess the performance of our retrieval system, we utilize a standardized benchmark of queries stored in a JSON file. Each entry in the dataset follows this structure:

* **`query`**: A string representing the textual modification (e.g., `"+glasses, -smiling"`).
* **`ground_truth`**: A dictionary where:
    * **Keys** are the indices of the **source images** used as the starting point.
    * **Values** are lists of indices for images considered valid retrievals for that specific source.

### Example Structure
```json
{
    "query": "+glasses, -smiling",
    "ground_truth": {
        "0": [1, 2, 3],
        "4": [5, 6, 7]
    }
}
```
In this example, image 0 serves as a source image (e.g., a smiling person without glasses). The system is expected to retrieve images 1, 2, or 3, which represent the "target" state (a non-smiling person with glasses), which should be visually similar to the source image but with the specified modifications applied.

In [ ]:
# Open the JSON file containing the benchmark annotations
annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# Print the number of annotations loaded
print(f"Loaded {len(annotations)} queries!")

We can define some utility functions to facilitate the evaluation process

In [ ]:
# Display a sample annotation to understand the structure of the data
print("Sample annotation shape", annotations[0].keys())

# Extract and print first text query
print("Text-Query example:", annotations[0].get("query", ""))

# Extract and print the source image ID for the first annotation
print("Source-Image example:", list(annotations[0].get("ground_truth", {}).keys())[:5],"...")

# Extract and print the list of ground truth indices for the first annotation
print("List of ground truth indices for the first annotation:", annotations[0].get("ground_truth", {}).get("13", [])[:5], "...")

def get_text_query(annotation: dict) -> str:
    """
    Helper function to extract the text query from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A string representing the text query (e.g., "+glasses, -smile").
    """
    return annotation.get("query", "")

def get_source_image_idxs(annotation: dict) -> list[int]:
    """
    Helper function to extract the source image IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A list of integers representing the source image IDs.
    """    
    # The key of the "ground_truth" must be converted to int since JSON keys are always strings
    return [int(key) for key in annotation.get("ground_truth", {}).keys()]

def get_ground_truth_indices(annotation: dict, source_image_idx: int) -> list[int]:
    """
    Helper function to extract the list of valid target IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
        - source_image_idx: The index of the source image for which to retrieve ground truth indices.
    Returns:
        - A list of valid target IDs (integers) that are considered correct matches for the given query.
    """
    return annotation.get("ground_truth", {}).get(str(source_image_idx), [])


In [ ]:
# Let's test these utility functions on the first annotation in the dataset
annotation = annotations[1]

text_query = get_text_query(annotation)
print("Text query:", text_query )

source_image_idx = get_source_image_idxs(annotation)[0]
print("Source image index:", source_image_idx)
plot_image_with_attributes(source_image_idx, figsize=(4, 4))

# Get the first 5 ground truth indices for this annotation and source image
ground_truth_indices = get_ground_truth_indices(annotation, source_image_idx)[:5]
print("Ground truth indices for this query:", ground_truth_indices)

plot_images(celeba, indices=ground_truth_indices, n_cols=5, n_rows=1, figsize=(10, 2))

### Evaluation Function

We evaluate the retrieval performance of each fusion mechanism on the benchmark dataset, comparing it against the baseline method.

We compute the recall and precision metrics for each source image in the query for `"K = {1, 5, 10}"`.
Then we average the result across all source images and keep track on each query separately.

In [ ]:
def evaluate( 
        fusion_mechanism: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
        query_embedding_function: Callable[[str, torch.nn.Module, CLIPProcessor, torch.device], list[torch.Tensor]],
        annotations: list[dict],
        model: torch.nn.Module,
        processor: CLIPProcessor,
        device: torch.device,
        embeddings: torch.Tensor
    ) -> list[dict]:
    '''
    Evaluates the retrieval performance of a given fusion mechanism on the benchmark dataset.
    Args:
    - fusion_mechanism: A function that takes in a text embedding and an image embedding and returns a unified query embedding (e.g., by summing them together).

    - query_embedding_function: A function that takes in a text query and returns its embedding using the CLIP model.
    - annotations: A list of benchmark annotations, where each annotation contains a text query, source image index, and a list of valid target indices.
    - model: The CLIP model to be used for computing text embeddings.
    - processor: The CLIP processor to be used for preprocessing text queries.
    - device: The device (CPU or GPU) on which the model is located.
    - embeddings: A tensor containing the pre-computed CLIP image embeddings for the entire dataset
    Returns:
    - A list of evaluation results for each query in the benchmark dataset, where each result contains the all list of evaluation metrics for each source image of the query.
    '''
    evaluation_results = []

    # For each annotation in the benchmark dataset
    for i, annotation in enumerate(annotations):
        print(f"Evaluating query Q{i+1}: {annotation.get('query', '')}")

        # Extract the text query and compute its embedding using the provided query_embedding_function
        text_query = get_text_query(annotation)
        embedded_text_query = query_embedding_function(text_query, model, processor, device)

        # Get the source image indices for this annotation
        query_image_idxs = get_source_image_idxs(annotation)

        query_evaluation_results = {}
        # Evaluate the retrieval performance for each source image
        for query_image_idx in query_image_idxs:
            
            # Compute the fused embedding
            embedded_query_image = embeddings[query_image_idx]
            fused_embedding = fusion_mechanism(embedded_text_query, embedded_query_image)
            
            # Normalize the fused embedding 
            fused_embedding = torch.nn.functional.normalize(fused_embedding, dim=0)

            # Compute the cosine similarity between the fused embedding and all image embeddings in the dataset
            # Equal to dot product since all embeddings are normalized to unit sphere
            similarities = torch.matmul(embeddings, fused_embedding)

            # Get the top 10 most similar images, excluding the source image itself
            _, topk_indices = torch.topk(similarities, k=11)
            topk_indices = topk_indices[topk_indices != query_image_idx][:10]
            retrieved_indices = topk_indices.tolist()

            # Evaluate the retrieval performance at K=1, 5, and 10 using the provided evaluation function
            image_query_eval_metrics = {}
            for k in [1, 5, 10]:
                eval_metrics = evaluate_retrieval(
                    retrieved_indices=retrieved_indices,
                    ground_truth_indices=get_ground_truth_indices(annotation, query_image_idx),
                    k=k
                )
                image_query_eval_metrics[k] = eval_metrics
            # Store the evaluation results for this source image
            query_evaluation_results[query_image_idx] = image_query_eval_metrics

        # Compute the average evaluation results across all source images for this query and store it in the final evaluation results list        
        evaluation_results.append(query_evaluation_results)
    return evaluation_results
        

def compute_query_average_results(query_evaluation_results: dict) -> dict:
    '''
    Computes the average Recall@K and Precision@K across all source images of a query for K=1, 5, and 10.
    Args:
    - query_evaluation_results: A dict mapping source image index to its evaluation metrics for K=1, 5, and 10.
    Returns:
    - A dictionary containing the average results for the query
    '''
    average_results = {}
    
    for k in [1, 5, 10]:
        # Compute the sum of Recall@K and Precision@K across all source images for this query
        recall_sum = 0
        precision_sum = 0
        # Get the number of test that we run for this query
        num_images = len(query_evaluation_results)

        # Iterate over the evaluation results for each source image and accumulate the Recall@K and Precision@K values
        for _, eval_metrics_per_k in query_evaluation_results.items():
            recall_sum += eval_metrics_per_k[k][f"Recall@{k}"]
            precision_sum += eval_metrics_per_k[k][f"Precision@{k}"]
        # Compute the average Recall@K and Precision@K for this query and store it in the average_results dictionary
        average_results[f"Recall@{k}"] = recall_sum / num_images
        average_results[f"Precision@{k}"] = precision_sum / num_images
        
        # Compute the 95% confidence intervals for Recall@K and Precision@K
        recall_std_error = np.sqrt((average_results[f"Recall@{k}"] * (1 - average_results[f"Recall@{k}"])) / num_images)
        precision_std_error = np.sqrt((average_results[f"Precision@{k}"] * (1 - average_results[f"Precision@{k}"])) / num_images)
        average_results[f"Recall@{k}_CI"] = 1.96 * recall_std_error
        average_results[f"Precision@{k}_CI"] = 1.96 * precision_std_error
    
    return average_results

## Baseline Method
To establish a baseline for our retrieval system, we evaluate a **zero-shot, training-free approach** that relies exclusively on CLIP embeddings and cosine similarity. 

The process is as follows:
* **Encode the source image** using the CLIP Vision Encoder to obtain its visual embedding.
* **Encode the query attributes** using the CLIP Text Encoder to obtain a text-based embedding.
* **Sum the two embeddings** to create a unified query representation:
* **Rank results** by computing the cosine similarity between this combined embedding and all candidate embeddings in the dataset.

For each **text query** (e.g., `"+glasses, -smiling"`), we parse the components, encode each of them using the CLIP Text Encoder, and sum them based on their signs with the source image embedding to form a unified query representation.

In [ ]:
def baseline_embedding(normalized_text_embedding: list[torch.Tensor], normalized_image_embedding: torch.Tensor, ALPHA: float = 1.0, BETA: float = 1.0) -> torch.Tensor:
    """
    Combine the query text and source image embeddings into a single query embedding.
    Parameters:
        - normalized_text_embedding: A list of normalized text embeddings for each component of the text query
        - normalized_image_embedding: A normalized image embedding for the source image
        - ALPHA: The weight for the text embedding in the fusion mechanism
        - BETA: The weight for the image embedding in the fusion mechanism
    Return:
        - A single query embedding that combines the text and image embeddings.
    """
    
    # If the text query has multiple components, we sum their embeddings together to get a single text embedding.
    if isinstance(normalized_text_embedding, list):
        normalized_text_embedding = torch.stack([embedding.view(-1) for embedding in normalized_text_embedding]).sum(dim=0)
    else:
        # If it's already a single embedding, we just need to flatten it to 1D
        normalized_text_embedding = normalized_text_embedding.view(-1)

    # Return the weighted sum of the normalized text embedding and the normalized image embedding as the final query embedding
    return ALPHA * normalized_text_embedding + BETA * normalized_image_embedding.view(-1)

def baseline_embed_query(text_query: str, model: torch.nn.Module, processor, device) -> list[torch.Tensor]:
    """
    Encode each text component into a normalized 1D tensor and keep its sign.
    """
    text_embeddings = []
    # Split the text query into components based on commas, and process each component separately
    for component in text_query.split(","):
        component = component.strip()
        sign = component[0]
        attr_name = component[1:].strip()

        # Encode each text component using the CLIP model to get its embedding
        inputs = processor(text=attr_name, return_tensors="pt").to(device)
        text_embedding = model.get_text_features(**inputs)

        if hasattr(text_embedding, "text_embeds"):
            text_embedding = text_embedding.text_embeds
        elif hasattr(text_embedding, "pooler_output"):
            text_embedding = text_embedding.pooler_output

        normalized_text_embedding = text_embedding.view(-1)
        normalized_text_embedding = normalized_text_embedding / normalized_text_embedding.norm(dim=-1, keepdim=True)

        # Depending on the sign, we either add or subtract the normalized text embedding to the list of text embeddings
        if sign == "+":
            text_embeddings.append(normalized_text_embedding)
        elif sign == "-":
            text_embeddings.append(-normalized_text_embedding)

    return text_embeddings

### Baseline Evaluation

In [ ]:
evaluation_results_baseline = evaluate(
    fusion_mechanism=baseline_embedding,
    query_embedding_function=baseline_embed_query,
    annotations=annotations,
    model=model,
    processor=processor,
    device=device,
    embeddings=embeddings
)
# Compute the average evaluation for each query
average_results_per_query_baseline = [compute_query_average_results(query_result) for query_result in evaluation_results_baseline]

In [ ]:
plot_metrics_across_k(average_results_per_query_baseline, title="Baseline Fusion Performance across K")

## First attempt: weighted fusion
As a first attempt to improve over the baseline, we experienced with fancier fusion mechanisms. The simplest approach that came to mind was to weight the text and image embeddings differently, instead of simply summing them.

The fusion mechanism can be defined as follows:
$$\text{QueryEmbedding} = \alpha \cdot \text{TextEmbedding} + \beta \cdot \text{ImageEmbedding}$$
Where:
- $\alpha$ is the weight for the text embedding in the fusion mechanism.
- $\beta$ is the weight for the image embedding in the fusion mechanism.

After some tries, we found that weighting the text more than the image (i.e., $\alpha > \beta$) tends to improve retrieval performance, suggesting that the text query provides more discriminative information for retrieval than the source image embedding.

In [ ]:
ALPHA = 0.7        # text query weight
BETA = 0.3         # image query weight
evaluation_results_weighted = evaluate(
    fusion_mechanism=partial(baseline_embedding, ALPHA=ALPHA, BETA=BETA),
    query_embedding_function=baseline_embed_query,
    annotations=annotations,
    model=model,
    processor=processor,
    device=device,
    embeddings=embeddings
)
# Compute the average evaluation for each query
average_results_per_query_weighted = [compute_query_average_results(query_result) for query_result in evaluation_results_weighted]

plot_metrics_across_k(average_results_per_query_weighted, title=f"Weighted Fusion Performance across K (ALPHA={ALPHA}, BETA={BETA})")

## Second Attempt: Prompt Engineering
Before arriving at our current strategy, we experimented with a purely linguistic approach.

### Handcrafted Antonyms
Initially, we attempted to solve CLIP's known struggles with negation by replacing negative signs with manually curated antonym phrases. For example, -smiling became "a person with a neutral expression" and -young became "old".

While this "humanized" the query, it introduced two significant issues:
1. **Over-specification**: Replacing "not smiling" with "neutral expression" was too restrictive. It inadvertently excluded other valid non-smiling states like frowning, shouting, or look of surprise.

2. **Token Leakage**: Phrases like "a person not wearing eyeglasses" still contained the "eyeglasses" token. CLIP’s attention mechanism often latched onto the object regardless of the "not," leading to confused embeddings.

### The Evolution: Going Back to Sign-Flipping
The previous approach replaced the baseline's sign flipping with concrete phrases. While intuitive, it discarded a core mathematical principle: **subtracting an attribute direction in embedding space removes everything correlated with that concept.** By switching to the current variant, we isolate the benefits of **phrase grounding** on positives without the performance degradation caused by linguistic antonyms.

### Core Design Choices

#### 1. Grounded Positive Phrasing
For every query component, we first resolve the **positive phrase** associated with the attribute. Instead of using the raw metadata tag (e.g., `Arched_Eyebrows`), we use a person-referring description like *"a person with arched eyebrows"*. This ensures that the base vector for any attribute is grounded in high-quality, descriptive language that CLIP understands well.

#### 2. Prompt-Template Ensembling
To reduce the variance and "noise" associated with any single sentence structure, we employ the zero-shot ensembling trick from the original CLIP paper. Each resolved phrase is rendered through a bank of templates:
* `"a photo of {phrase}"`
* `"a portrait of {phrase}"`
* `"a cropped photo of {phrase}"`
* ...and several others.

Each version is encoded, L2-normalized, and then averaged. The final mean vector is re-normalized to create a robust, stable representation of that specific attribute.

#### 3. Mathematical Negation for `-` Components
This is where we depart from the antonym approach. When a query requires a negative attribute (e.g., `-Smiling`):
* We **do not** look up an antonym phrase.
* We **do not** use "not" in the prompt.
* Instead, we take the ensembled embedding of the **positive** phrase ("a person smiling") and **negate the resulting vector** (multiply by -1).

### Why This Performs Better
* **Avoids Over-specification:** Negating the "smiling" vector removes the concept of smiling generally, correctly capturing faces that are frowning, surprised, or shouting—states that a "neutral expression" prompt would miss.
* **Bypasses "Not" Issues:** CLIP's text encoder is notoriously weak at handling negation tokens. By performing negation in the embedding space (latent space), we bypass the linguistic confusion entirely.
* **Clean Subtraction:** This maintains the "attribute direction" logic, ensuring that the model focuses on the absence of the feature rather than the presence of a different, specific feature.

> **Summary:** By anchoring our vectors in humanized positive descriptions and then applying mathematical sign-flipping for negatives, we achieve a retrieval system that is both linguistically rich and mathematically sound.

In [ ]:
# Person-referring positive phrases for each CelebA attribute.
# Polarity is handled in embedding space (sign-flip), so we only store positives.
humanized_mappings = {
    "5_o_Clock_Shadow":     ["a person with a 5 o'clock shadow", "a person with light facial stubble", "a person with short beard stubble"],
    "Arched_Eyebrows":      ["a person with arched eyebrows", "a person with curved eyebrows"],
    "Attractive":           ["an attractive person", "a good-looking person", "a visually appealing person"],
    "Bags_Under_Eyes":      ["a person with bags under the eyes", "a person with eye bags", "a tired-looking person with under-eye bags"],
    "Bald":                 ["a bald person", "a person with no hair", "a person with a shaved head"],
    "Bangs":                ["a person with bangs", "a person with fringe hair"],
    "Big_Lips":             ["a person with full lips", "a person with big lips"],
    "Big_Nose":             ["a person with a big nose", "a person with a large nose"],
    "Black_Hair":           ["a person with black hair", "a person with dark black hair"],
    "Blond_Hair":           ["a person with blond hair", "a person with blonde hair", "a person with light blonde hair"],
    "Blurry":               ["a blurry photo of a person", "an out-of-focus image of a person", "a blurred image of a person"],
    "Brown_Hair":           ["a person with brown hair", "a person with dark brown hair"],
    "Bushy_Eyebrows":       ["a person with bushy eyebrows", "a person with thick eyebrows"],
    "Chubby":               ["a chubby person", "a person with a round face"],
    "Double_Chin":          ["a person with a double chin", "a person with a noticeable double chin"],
    "Eyeglasses":           ["a person wearing eyeglasses", "a person wearing glasses", "a person with glasses", "a face with eyeglasses"],
    "Goatee":               ["a person with a goatee", "a person with a goatee beard"],
    "Gray_Hair":            ["a person with gray hair", "a person with grey hair", "a person with silver hair"],
    "Heavy_Makeup":         ["a person wearing heavy makeup", "a person with noticeable makeup", "a person with strong makeup"],
    "High_Cheekbones":      ["a person with high cheekbones", "a person with prominent cheekbones"],
    "Male":                 ["a man", "a male person"],
    "Mouth_Slightly_Open":  ["a person with their mouth slightly open", "a person with slightly open lips"],
    "Mustache":             ["a person with a mustache", "a person with facial hair and a mustache"],
    "Narrow_Eyes":          ["a person with narrow eyes", "a person with small eyes"],
    "No_Beard":             ["a clean-shaven person", "a person without a beard", "a person with no facial hair"],
    "Oval_Face":            ["a person with an oval face", "a person with an oval-shaped face"],
    "Pale_Skin":            ["a person with pale skin", "a person with light skin tone"],
    "Pointy_Nose":          ["a person with a pointy nose", "a person with a sharp nose"],
    "Receding_Hairline":    ["a person with a receding hairline", "a person with thinning hairline"],
    "Rosy_Cheeks":          ["a person with rosy cheeks", "a person with flushed cheeks"],
    "Sideburns":            ["a person with sideburns", "a person with long sideburns"],
    "Smiling":              ["a smiling person", "a person who is smiling", "a person with a happy expression", "a person with a big smile"],
    "Straight_Hair":        ["a person with straight hair", "a person with smooth straight hair"],
    "Wavy_Hair":            ["a person with wavy hair", "a person with curly wavy hair"],
    "Wearing_Earrings":     ["a person wearing earrings", "a person with earrings"],
    "Wearing_Hat":          ["a person wearing a hat", "a person with a hat"],
    "Wearing_Lipstick":     ["a person wearing lipstick", "a person with lipstick"],
    "Wearing_Necklace":     ["a person wearing a necklace", "a person with a necklace"],
    "Wearing_Necktie":      ["a person wearing a necktie", "a person with a tie"],
    "Young":                ["a young person", "a youthful person", "a person who looks young"],
}


In [ ]:
# CLIP-style prompt templates used for ensembling. Each expects a "{phrase}" slot
# that fills in with a complete person-referring phrase, e.g. "a smiling person".
prompt_templates = [
    "a photo of {phrase}.",
    "a portrait of {phrase}.",
    "a picture of {phrase}.",
    "an image of {phrase}.",
    "a close-up photo of {phrase}.",
    "a cropped photo of {phrase}.",
    "a headshot of {phrase}.",
    "a high resolution photo of {phrase}.",
    "a candid photo of {phrase}.",
    "a studio portrait of {phrase}.",
]

@torch.no_grad()
def _encode_prompt_ensemble(prompts: list[str], model: torch.nn.Module, processor, device) -> torch.Tensor:
    """
    Encode each prompt in the list, normalize, compute the mean embedding, and re-normalize.
    Parameters:
        - prompts: A list of strings, each being a different prompt template filled with the same phrase (e.g. "a photo of a smiling person", "a portrait of a smiling person", etc.)
        - model: The CLIP model to be used for encoding the prompts.
        - processor: The CLIP processor to be used for preprocessing the prompts.
        - device: The device (CPU or GPU) on which the model is located.
    Returns:
        - A single 1-D tensor representing the mean-pooled, normalized embedding of the prompts.
    """
    embeddings = []

    # For each prompt in the list
    for prompt in prompts:
        # Encode the prompt using the CLIP model to get its embedding
        inputs = processor(text=prompt, return_tensors="pt").to(device)
        embedding = model.get_text_features(**inputs)
        if hasattr(embedding, "text_embeds"):
            embedding = embedding.text_embeds
        elif hasattr(embedding, "pooler_output"):
            embedding = embedding.pooler_output

        # Normalize the embedding to unit sphere and flatten to 1D
        embedding = embedding.view(-1)
        normalized_embedding = embedding / embedding.norm(dim=-1, keepdim=True)
        embeddings.append(normalized_embedding)

    # Compute the mean embedding across all prompts, and re-normalize it to get the final embedding for this text component
    mean_embedding = torch.stack(embeddings).mean(dim=0)
    return mean_embedding / mean_embedding.norm(dim=-1, keepdim=True)


def typed_signflip_embed_query(text_query: str, model: torch.nn.Module, processor, device) -> list[torch.Tensor]:
    """
    Hybrid encoder: humanized positive phrases with template ensembling, then
    sign-flip in embedding space for '-' components (baseline-style negation).

    For each parsed component (e.g. "+Eyeglasses", "-Smiling"):
      1. Look up the list of positive paraphrases for the attribute.
      2. Render every (paraphrase x template) pair and ensemble-encode in one pass.
      3. Append the embedding as-is for '+' components, negated for '-' components.
    """
    text_embeddings = []
    # For each attribute in the text query
    for component in text_query.split(","):
        component = component.strip()
        # Get the sign and attribute name
        sign, attr_name = component[0], component[1:].strip()

        # Get the list of humanized phrases for this attribute
        phrases = humanized_mappings[attr_name]
        prompts = [template.format(phrase=phrase) for phrase in phrases for template in prompt_templates]
        normalized_embedding = _encode_prompt_ensemble(prompts, model, processor, device)

        if sign == "+":
            text_embeddings.append(normalized_embedding)
        elif sign == "-":
            text_embeddings.append(-normalized_embedding)

    return text_embeddings

In [ ]:
evaluation_results_typed = evaluate(
    fusion_mechanism=baseline_embedding,
    query_embedding_function=typed_signflip_embed_query,
    annotations=annotations,
    model=model,
    processor=processor,
    device=device,
    embeddings=embeddings,
)

# Compute the average evaluation for each query
average_results_per_query_typed = [
    compute_query_average_results(query_result) for query_result in evaluation_results_typed
]

# Plot the results
plot_metrics_across_k(
    average_results_per_query_typed,
    title=f"Prompt-Engineering + Sign-Flip Fusion Performance across K",
)

In [ ]:
plot_methods_comparison(
    {
        "Baseline":         average_results_per_query_baseline,
        "Weighted Fusion":  average_results_per_query_weighted,
        "Prompt Engineering": average_results_per_query_typed
    },
    title="Method Comparison — per-query Recall@K and Precision@K",
)